In [1]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 
import glob

In [ ]:
# find . -type f | sort

In [2]:
study_files = {
    "Choudhury2022": {
        "matrix": "Data_Choudhury2022_Brain/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Choudhury2022_Brain/Genes.txt",
        "cells":  "Data_Choudhury2022_Brain/Cells.csv",
        "samples": "Data_Choudhury2022_Brain/Samples.csv",
        "units": "UMI"
    },
    "Couturier2020": {
        "matrix": "Data_Couturier2020_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Couturier2020_Brain/Genes.txt",
        "cells":  "Data_Couturier2020_Brain/Cells.csv",
        "samples": "Data_Couturier2020_Brain/Samples.csv",
        "units": "TPM"
    },
    "Darmanis2017": {
        "matrix": "Data_Darmanis2017_Brain/normalized_Exp_data_TPM.mtx",
        "genes":  "Data_Darmanis2017_Brain/Genes.txt",
        "cells":  "Data_Darmanis2017_Brain/Cells.csv",
        "samples": "Data_Darmanis2017_Brain/Samples.csv",
        "units": "TPM"          # normalized TPM is still TPM
    },
    "Filbin2018": {
        "matrix": "Data_Filbin2018_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Filbin2018_Brain/Genes.txt",
        "cells":  "Data_Filbin2018_Brain/Cells.csv",
        "samples": "Data_Filbin2018_Brain/Samples.csv",
        "units": "TPM"
    },
    "Gojo2020": {
        "matrix": "Data_Gojo2020_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Gojo2020_Brain/Genes.txt",
        "cells":  "Data_Gojo2020_Brain/Cells.csv",
        "samples": "Data_Gojo2020_Brain/Samples.csv",
        "units": "TPM"
    },
    "Hovestadt2019": {
        "matrix": "Data_Hovestadt2019_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Hovestadt2019_Brain/Genes.txt",
        "cells":  "Data_Hovestadt2019_Brain/Cells.csv",
        "samples": "Data_Hovestadt2019_Brain/Samples.csv",
        "units": "TPM"
    },
    "Neftel2019_10X": {
        "matrix": "Data_Neftel2019_Brain/10X/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Neftel2019_Brain/10X/Genes.txt",
        "cells":  "Data_Neftel2019_Brain/10X/Cells.csv",
        "samples": "Data_Neftel2019_Brain/Samples.csv",
        "units": "UMI"
    },
    "Neftel2019_SmartSeq2": {
        "matrix": "Data_Neftel2019_Brain/SmartSeq2/Exp_data_TPM.mtx",
        "genes":  "Data_Neftel2019_Brain/SmartSeq2/Genes.txt",
        "cells":  "Data_Neftel2019_Brain/SmartSeq2/Cells.csv",
        "samples": "Data_Neftel2019_Brain/Samples.csv",
        "units": "TPM"
    },
    "Tirosh2016": {
        "matrix": "Data_Tirosh2016_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Tirosh2016_Brain/Genes.txt",
        "cells":  "Data_Tirosh2016_Brain/Cells.csv",
        "samples": "Data_Tirosh2016_Brain/Samples.csv",
        "units": "TPM"
    },
    "Venteicher2017": {
        "matrix": "Data_Venteicher2017_Brain/Exp_data_TPM.mtx",
        "genes":  "Data_Venteicher2017_Brain/Genes.txt",
        "cells":  "Data_Venteicher2017_Brain/Cells.csv",
        "samples": "Data_Venteicher2017_Brain/Samples.csv",
        "units": "TPM"
    },
    "Wang2019": {
        "matrix": "Data_Wang2019_Brain/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Wang2019_Brain/Genes.txt",
        "cells":  "Data_Wang2019_Brain/Cells.csv",
        "samples": "Data_Wang2019_Brain/Samples.csv",
        "units": "UMI"
    },
    "Yuan2018": {
        "matrix": "Data_Yuan2018_Brain/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Yuan2018_Brain/Genes.txt",
        "cells":  "Data_Yuan2018_Brain/Cells.csv",
        "samples": "Data_Yuan2018_Brain/Samples.csv",
        "units": "UMI"
    }
}

In [3]:
study_name = 'Choudhury2022'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 58,843 cells × 33,538 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Meningioma']
After gene filter: 54,675 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.55
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 18.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 102.3 seconds
After doublet removal: 54,640 cells
After MT filter: 46,006 cells
Normalised (UMI) – max value 13.46


In [4]:
study_name = 'Couturier2020'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 100,335 cells × 33,694 genes
   cell_type: 81792 NaN
   cell_type: 81792 suspicious
Flagged 81792 cells
After metadata drop: 18,543 cells
   Added cancer_type column. Unique values: ['Glioblastoma' 'Normal']
After gene filter: 18,404 cells
Preprocessing...


/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/scrublet/helper_functions.py:239: RuntimeWarning: invalid value encountered in log
  gLog = lambda input: np.log(input[1] * np.exp(-input[0]) + input[2])
/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/scrublet/helper_functions.py:252: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.67
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 16.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 22.3 seconds
After doublet removal: 18,401 cells
After MT filter: 18,401 cells
Normalised (TPM) – max value 2.82


In [5]:
study_name = 'Darmanis2017'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 3,589 cells × 23,368 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Glioblastoma']
After gene filter: 3,541 cells
Preprocessing...


/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/scrublet/helper_functions.py:239: RuntimeWarning: invalid value encountered in log
  gLog = lambda input: np.log(input[1] * np.exp(-input[0]) + input[2])
/home/pandeyps/.pyenv/versions/3.11.9/lib/python3.11/site-packages/scrublet/helper_functions.py:252: RuntimeWarning: invalid value encountered in sqrt
  CV_input = np.sqrt(b);


Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.47
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 49.1%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.3%
Elapsed time: 3.4 seconds
After doublet removal: 3,536 cells
After MT filter: 3,536 cells
Normalised (TPM) – max value 2.98


In [6]:
study_name = 'Filbin2018'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 2,587 cells × 23,686 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['H3K27M Glioma' 'Normal']
After gene filter: 1,814 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.39
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 40.3%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 1.9 seconds
After doublet removal: 1,813 cells
After MT filter: 1,813 cells
Normalised (TPM) – max value 12.80


In [7]:
study_name = 'Gojo2020'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 6,739 cells × 20,447 genes
   cell_type: 614 NaN
   cell_type: 614 suspicious
Flagged 614 cells
After metadata drop: 6,125 cells
   Added cancer_type column. Unique values: ['Ependymoma']
After gene filter: 4,988 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.52
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 25.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 5.2 seconds
After doublet removal: 4,985 cells
After MT filter: 4,985 cells
Normalised (TPM) – max value 13.29


In [8]:
study_name = 'Hovestadt2019'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 8,691 cells × 23,686 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Medulloblastoma']
After gene filter: 7,133 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.57
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 13.9%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 7.8 seconds
After doublet removal: 7,133 cells
After MT filter: 7,133 cells
Normalised (TPM) – max value 12.97


In [9]:
study_name = 'Neftel2019_10X'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 16,201 cells × 30,401 genes
   cell_type: 1152 NaN
   cell_type: 1152 suspicious
Flagged 1152 cells
After metadata drop: 15,049 cells
   Added cancer_type column. Unique values: ['Glioblastoma']
After gene filter: 13,390 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.41
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 17.0%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 1.1%
Elapsed time: 14.9 seconds
After doublet removal: 13,366 cells
After MT filter: 13,214 cells
Normalised (UMI) – max value 13.36


In [10]:
study_name = 'Neftel2019_SmartSeq2'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 7,930 cells × 23,686 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Glioblastoma']
After gene filter: 4,593 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.51
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 31.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 5.6 seconds
After doublet removal: 4,592 cells
After MT filter: 4,592 cells
Normalised (TPM) – max value 13.10


In [11]:
study_name = 'Tirosh2016'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 4,347 cells × 23,686 genes
   cell_type: 23 NaN
   cell_type: 23 suspicious
Flagged 23 cells
After metadata drop: 4,324 cells
   Added cancer_type column. Unique values: ['Oligodendroglioma']
After gene filter: 3,499 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.48
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 26.4%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.4%
Elapsed time: 3.9 seconds
After doublet removal: 3,495 cells
After MT filter: 3,495 cells
Normalised (TPM) – max value 13.08


In [12]:
study_name = 'Venteicher2017'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 6,341 cells × 23,686 genes
   cell_type: 98 NaN
   cell_type: 98 suspicious
Flagged 98 cells
After metadata drop: 6,243 cells
   Added cancer_type column. Unique values: ['Astrocytoma']
After gene filter: 3,270 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.48
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 43.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.0%
Elapsed time: 3.4 seconds
After doublet removal: 3,270 cells
After MT filter: 3,270 cells
Normalised (TPM) – max value 13.09


In [13]:
study_name = 'Wang2019'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 16,317 cells × 33,545 genes
Flagged 0 cells
   Added cancer_type column. Unique values: ['Glioblastoma']
After gene filter: 15,911 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.51
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 24.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.8%
Elapsed time: 18.7 seconds
After doublet removal: 15,880 cells
After MT filter: 15,261 cells
Normalised (UMI) – max value 12.67


In [14]:
study_name = 'Yuan2018'
paths = study_files[study_name]

# 1. Load matrix and transpose
adata = sc.read_mtx(paths['matrix']).T

# 2. Gene names
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()   # added for safety

# 3. Cell metadata – fix barcode type before assignment
cells = pd.read_csv(paths["cells"], index_col=0)
cells.index = cells.index.astype(str)
adata.obs_names = cells.index
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name
print(f"Loaded: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

# 4. Drop NaN / suspicious metadata
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease']
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]
drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask |= nan_mask
    if vals.dtype == object:
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} suspicious")
            drop_mask |= suspect

print(f"Flagged {drop_mask.sum()} cells")
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"After metadata drop: {adata.n_obs:,} cells")

# 5. Add cancer_type column (if missing)
if 'cancer_type' not in adata.obs.columns:
    samples = pd.read_csv(paths["samples"], index_col=0)
    if 'cancer_type' in samples.columns:
        samples_unique = samples[~samples.index.duplicated(keep='first')]
        sample_to_cancer = samples_unique['cancer_type'].to_dict()
        adata.obs['cancer_type'] = adata.obs['sample'].map(sample_to_cancer)
    else:
        adata.obs['cancer_type'] = 'Brain'   # fallback
    print(f"   Added cancer_type column. Unique values: {adata.obs['cancer_type'].unique()}")

# 6. Gene‑count filter
sc.pp.calculate_qc_metrics(adata, inplace=True)
keep = (adata.obs["n_genes_by_counts"] >= 200) & (adata.obs["n_genes_by_counts"] <= 6000)
adata = adata[keep, :].copy()
print(f"After gene filter: {adata.n_obs:,} cells")

# 7. Doublet removal
counts = adata.X.toarray() if hasattr(adata.X, 'toarray') else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()
print(f"After doublet removal: {adata.n_obs:,} cells")

# 8. Mitochondrial filter
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
keep_mt = adata.obs['pct_counts_mt'] <= 15
adata = adata[keep_mt, :].copy()
print(f"After MT filter: {adata.n_obs:,} cells")

# 9. Normalisation
unit = study_files[study_name]["units"]
if unit == "TPM":
    sc.pp.log1p(adata)
else:   # UMI
    sc.pp.normalize_total(adata, target_sum=1e6)   # CPM
    sc.pp.log1p(adata)
print(f"Normalised ({unit}) – max value {adata.X.max():.2f}")

for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

out_file = f"{study_name}.h5ad"
adata.write_h5ad(out_file)

Loaded: 23,793 cells × 58,828 genes
Flagged 0 cells
   Added cancer_type column. Unique values: [nan 'Glioblastoma']
After gene filter: 23,750 cells
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.71
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 3.7%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 27.8 seconds
After doublet removal: 23,749 cells
After MT filter: 23,749 cells
Normalised (UMI) – max value 12.12


In [15]:
INPUT_PATTERN = "*.h5ad"   

files = sorted(glob.glob(INPUT_PATTERN))

for f in files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0] if 'study' in adata.obs.columns else f
    print(f"{study_name}  –  {adata.n_obs:,} cells, {adata.n_vars:,} genes")
    
    for col in sorted(adata.obs.columns):
        if pd.api.types.is_numeric_dtype(adata.obs[col]):
            continue
        
        vals = adata.obs[col].dropna().unique()
        n_unique = len(vals)
        
        if n_unique <= 25:
            val_str = ', '.join(str(v) for v in sorted(vals))
        else:
            val_str = f"{n_unique} unique values – first 15: " + ', '.join(str(v) for v in sorted(vals)[:15])
        
        print(f" {col}: {val_str}")
    
    print()

Choudhury2022  –  46,006 cells, 33,538 genes
 cancer_type: Meningioma
 cell_cycle_phase: G1/S, G2/M, Intermediate, Not cycling, nan
 cell_subtype: Active monocytes, CD163 monocytes, CD4 T cells, CD8 T cells, Cycling G2M phase meningioma cells 1, Cycling G2M phase meningioma cells 2, Cycling S phase meningioma cells, ECM remodeling meningioma cells, G1 phase meningioma cells, Glia, Meningioma cells, Monocytes, Pericytes, SSTR2A meningioma cells, Vascular cells
 cell_type: Endothelial, Glia, Malignant, Monocyte, Pericyte, T_cell
 mp_assignment: 34 unique values – first 15: CD4 - Cell Cycle, CD4 - Cytotoxic, CD4 - Interferon, CD4 - Naive1, CD4 - Stress/HSP, CD4 - Treg, CD8 - Chromatin, CD8 - Cytotoxic, CD8 - Heat shock, CD8 - Memory/Naive1, CD8 - Naive2, CD8 - Naive3, CD8 - Unassigned2, Cell Cycle, Cell Cycle - G1/S
 mp_top: 77 unique values – first 15: Alveolar, Astrocytes, CD4 - Cell Cycle, CD4 - Cytotoxic, CD4 - Dysfunction, CD4 - Glycolysis/MYC, CD4 - Interferon, CD4 - Naive1, CD4 - N

In [16]:
INPUT_PATTERN = "*.h5ad"  
OUTPUT_FILE = "brain.h5ad"

valid_prefixes = [
    'Choudhury2022', 'Couturier2020', 'Darmanis2017', 'Filbin2018',
    'Gojo2020', 'Hovestadt2019', 'Neftel2019_10X', 'Neftel2019_SmartSeq2',
    'Tirosh2016', 'Venteicher2017', 'Wang2019', 'Yuan2018'
]
all_files = sorted(glob.glob(INPUT_PATTERN))
real_files = [f for f in all_files if any(f.startswith(p) for p in valid_prefixes)]
print(f"Found {len(real_files)} brain study files (ignored {len(all_files)-len(real_files)} other files)\n")

def label_malignant(adata, study_name):
    if 'malignant' in adata.obs.columns:
        mal_col = adata.obs['malignant'].astype(str).str.lower().str.strip()
        mask = mal_col == 'yes'
        if mask.any():
            return mask, "malignant column (yes/no)"

    if 'cell_type' in adata.obs.columns:
        ct = adata.obs['cell_type'].astype(str).str.lower().str.strip()
        if ct.str.contains('malignant', na=False).any():
            return ct.str.contains('malignant', na=False), "cell_type='Malignant'"
    if 'cancer_type' in adata.obs.columns:
        ct = adata.obs['cancer_type'].astype(str).str.lower().str.strip()
        if not ct.str.contains('normal', na=False).any():
            return pd.Series(True, index=adata.obs.index), "all cells assumed malignant (tumor study)"

    return pd.Series(False, index=adata.obs.index), "all non‑malignant"

temp_files = []
total_malig = 0
total_non = 0

for f in real_files:
    adata = sc.read_h5ad(f)
    study_name = adata.obs['study'].iloc[0]

    if 'cancer_type' not in adata.obs.columns or adata.obs['cancer_type'].isna().all():
        adata.obs['cancer_type'] = 'Brain Cancer'   # safe fallback
        print(f"  {study_name}: cancer_type missing/NaN – set to 'Brain Cancer'")
    else:
        most_common = adata.obs['cancer_type'].mode()[0] if not adata.obs['cancer_type'].mode().empty else 'Brain Cancer'
        adata.obs['cancer_type'] = adata.obs['cancer_type'].fillna(most_common)

    malignant_mask, strategy = label_malignant(adata, study_name)
    adata.obs['is_malignant'] = malignant_mask.astype(int)

    malig_count = malignant_mask.sum()
    non_count = (~malignant_mask).sum()
    total_malig += malig_count
    total_non += non_count

    print(f"{study_name:<25s}  {adata.n_obs:>7,} cells  →  malignant: {malig_count:>6,}   non‑malignant: {non_count:>7,}   ({strategy})")

    temp_file = f"__temp_{study_name}.h5ad"
    adata.write_h5ad(temp_file)
    temp_files.append(temp_file)

print(f"\n{'='*60}")
print(f"TOTAL: {total_malig+total_non:>7,} cells  →  malignant: {total_malig:>6,}   non‑malignant: {total_non:>7,}")

print(f"\nMerging {len(temp_files)} studies with inner gene join...")
adatas = [sc.read_h5ad(tf) for tf in temp_files]
adata_all = adatas[0].concatenate(
    adatas[1:],
    batch_key='study',
    join='inner',
    index_unique='-'
)
print(f"Merged: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")

study_names_original = []
for tf in temp_files:
    ad = sc.read_h5ad(tf)
    study_names_original.append(ad.obs['study'].iloc[0])
study_map = {str(i): name for i, name in enumerate(study_names_original)}
adata_all.obs['study'] = adata_all.obs['study'].astype(str).map(study_map)
print(f"Study names restored: {sorted(adata_all.obs['study'].unique())}")

essential = ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']
keep = [c for c in essential if c in adata_all.obs.columns]
adata_all.obs = adata_all.obs[keep]
print(f"Metadata retained: {list(adata_all.obs.columns)}")


print("\nCancer types:", sorted(adata_all.obs['cancer_type'].astype(str).unique()))
print("Malignant distribution:")
print(adata_all.obs['is_malignant'].value_counts())

# Drop NaN if any
for col in keep:
    n_nan = adata_all.obs[col].isna().sum()
    if n_nan > 0:
        print(f" {col}: {n_nan} NaN – dropping")
        adata_all = adata_all[~adata_all.obs[col].isna(), :].copy()

for col in adata_all.obs.columns:
    if adata_all.obs[col].dtype == 'object':
        adata_all.obs[col] = adata_all.obs[col].astype(str)

print(f"Final clean dataset: {adata_all.n_obs:,} cells × {adata_all.n_vars:,} genes")
adata_all.write_h5ad(OUTPUT_FILE)

# Clean up temp files
for tf in temp_files:
    os.remove(tf)

Found 12 brain study files (ignored 0 other files)

Choudhury2022               46,006 cells  →  malignant: 27,163   non‑malignant:  18,843   (cell_type='Malignant')
Couturier2020               18,401 cells  →  malignant: 18,401   non‑malignant:       0   (cell_type='Malignant')
Darmanis2017                 3,536 cells  →  malignant:  1,056   non‑malignant:   2,480   (cell_type='Malignant')
Filbin2018                   1,813 cells  →  malignant:  1,531   non‑malignant:     282   (cell_type='Malignant')
Gojo2020                     4,985 cells  →  malignant:  4,439   non‑malignant:     546   (cell_type='Malignant')
Hovestadt2019                7,133 cells  →  malignant:  7,133   non‑malignant:       0   (all cells assumed malignant (tumor study))
Neftel2019_10X              13,214 cells  →  malignant:  9,359   non‑malignant:   3,855   (cell_type='Malignant')
Neftel2019_SmartSeq2         4,592 cells  →  malignant:  3,608   non‑malignant:     984   (malignant column (yes/no))
Tirosh2016  

/tmp/ipykernel_14125/788982449.py:65: FutureWarning: The method concatenate is deprecated and will be removed in the future. Use anndata.concat instead of AnnData.concatenate. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  adata_all = adatas[0].concatenate(


Merged: 145,455 cells × 16,495 genes
Study names restored: ['Choudhury2022', 'Couturier2020', 'Darmanis2017', 'Filbin2018', 'Gojo2020', 'Hovestadt2019', 'Neftel2019_10X', 'Neftel2019_SmartSeq2', 'Tirosh2016', 'Venteicher2017', 'Wang2019', 'Yuan2018']
Metadata retained: ['sample', 'cell_type', 'cancer_type', 'study', 'is_malignant']

Cancer types: ['Astrocytoma', 'Ependymoma', 'Glioblastoma', 'H3K27M Glioma', 'Medulloblastoma', 'Meningioma', 'Normal', 'Oligodendroglioma', 'nan']
Malignant distribution:
is_malignant
1    107371
0     38084
Name: count, dtype: int64
 cell_type: 7133 NaN – dropping
Final clean dataset: 138,322 cells × 16,495 genes
